In [3]:
# -*- coding: utf-8 -*-
"""
基于城市行政边界与时间窗口的 GPM 0.5h 【格网级 (Gridded)】降雨提取脚本
数据源: NASA GPM IMERG V06 (半小时, 0.1度约10km)
输出格式: 包含每个像元中心经纬度、时间、降雨量的长表格CSV
"""

import os
import pandas as pd
import geopandas as gpd
import ee
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# =========================================================
# 0. 网络代理配置 (根据你的梯子软件修改端口)
# =========================================================
os.environ['HTTP_PROXY'] = 'http://YOUR_PROXY_HOST:PORT'
os.environ['HTTPS_PROXY'] = 'http://YOUR_PROXY_HOST:PORT'

# =========================================================
# 1. GEE 初始化
# =========================================================
try:
    ee.Initialize()
    print("GEE 初始化成功！")
except Exception as e:
    print("正在进行 GEE 认证...")
    ee.Authenticate()
    ee.Initialize()
    print("GEE 授权并初始化成功！")

# =========================================================
# 2. 路径配置
# =========================================================
# 上一步生成的干净数据
CSV_PATH = r"D:/SM/zdxrun/datatest/research/数据集/screening_outputs/spatial_checks/Weibo_Flood_Master_V4_SpatialCleaned.csv"
SHP_PATH = r"D:\数据\2023年中国省市县三级行政区划shp\2023年中国省市县三级行政区划shp\2023年地级\2023年初地级矢量.shp"

BASE_DIR = Path(r"D:/SM/zdxrun/datatest/research/数据集")
OUT_DIR = BASE_DIR / "rainfall_gridded_05h"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# 3. 读取事件窗口
# =========================================================
print("\n正在读取事件表并计算时间窗口...")
df = pd.read_csv(CSV_PATH, low_memory=False)

df['开始时间'] = pd.to_datetime(df['开始时间'], errors='coerce')
df['used_time_str'] = pd.to_datetime(df['used_time_str'], errors='coerce')

event_windows = df.groupby(['city_clean', 'Event_ID']).agg(
    start_time_bjt=('开始时间', 'min'),
    end_time_bjt=('used_time_str', 'max'),
    n_points=('clean_lon', 'count')
).reset_index()

event_windows = event_windows.dropna(subset=['start_time_bjt', 'end_time_bjt'])
event_windows = event_windows[event_windows['Event_ID'] == '1283'].copy()

event_windows['start_time_bjt'] = event_windows['start_time_bjt'] - pd.Timedelta(hours=48)
event_windows['end_time_bjt'] = event_windows['end_time_bjt'] + pd.Timedelta(hours=12)

# BJT 转换为 UTC，供 GEE 调用
event_windows['start_time_utc'] = event_windows['start_time_bjt'] - pd.Timedelta(hours=8)
event_windows['end_time_utc'] = event_windows['end_time_bjt'] - pd.Timedelta(hours=8)


print(f"测试模式：只提取 Event 1283，共 {len(event_windows)} 个事件。")
print(f"成功构建 {len(event_windows)} 个事件的时间窗口。")

# =========================================================
# 4. 加载行政边界 SHP
# =========================================================
print("正在加载全国省市县行政区划矢量数据(shp)...")
try:
    china_shp = gpd.read_file(SHP_PATH, encoding='utf-8')
except Exception:
    china_shp = gpd.read_file(SHP_PATH, encoding='gbk')

if china_shp.crs.to_string() != 'EPSG:4326':
    china_shp = china_shp.to_crs(epsg=4326)

has_diji = '地级' in china_shp.columns
has_shengji = '省级' in china_shp.columns

# =========================================================
# 5. 定义格网降雨提取核心函数
# =========================================================
def extract_gpm_gridded(start_utc, end_utc, city_poly):
    """
    针对单个城市，提取 GPM 0.5h 各格网降雨量
    """
    simplified_poly = city_poly.simplify(0.01)
    if simplified_poly.geom_type == 'Polygon':
        coords =[list(simplified_poly.exterior.coords)]
        ee_geom = ee.Geometry.Polygon(coords)
    elif simplified_poly.geom_type == 'MultiPolygon':
        coords = [[list(poly.exterior.coords)] for poly in simplified_poly.geoms]
        ee_geom = ee.Geometry.MultiPolygon(coords)
    else:
        raise ValueError("不支持的几何类型")

    # 定义影像集
    dataset = ee.ImageCollection("NASA/GPM_L3/IMERG_V06") \
                .filterBounds(ee_geom) \
                .filterDate(start_utc.strftime('%Y-%m-%dT%H:%M:%S'), 
                            end_utc.strftime('%Y-%m-%dT%H:%M:%S')) \
                .select('precipitationCal')

    # 【关键操作】：裁剪(Clip)。将边界外的像素置为 NULL，方便后续清洗
    dataset_clipped = dataset.map(lambda img: img.clip(ee_geom))

    # 使用 getRegion 获取每个像元中心点的经纬度和对应的值
    # 参数 10000 代表空间分辨率 10,000米 (10km)
    #region_data = dataset_clipped.getRegion(geometry=ee_geom, scale=10000).getInfo()
    
    # getRegion 返回的是一个二维列表，第一行是表头
    if not region_data or len(region_data) <= 1:
        return pd.DataFrame()
    
    header = region_data[0]
    data_rows = region_data[1:]
    df_res = pd.DataFrame(data_rows, columns=header)
    
    # 清洗数据
    # getRegion 会返回外接矩形内的所有像元，被裁剪到边界外的像元 precipitationCal 为 NaN
    df_res = df_res.dropna(subset=['precipitationCal']).copy()
    
    if df_res.empty:
        return df_res
        
    # 时间戳转换
    df_res['Datetime_UTC'] = pd.to_datetime(df_res['time'], unit='ms')
    df_res['Datetime_BJT'] = df_res['Datetime_UTC'] + pd.Timedelta(hours=8)
    
    # 单位转换：mm/h 转 mm/0.5h
    df_res['Rainfall_Mean_mmh'] = df_res['precipitationCal']
    df_res['Rainfall_05h_mm'] = df_res['Rainfall_Mean_mmh'] / 2.0
    
    # 提取需要的列并重命名经纬度
    cols_to_keep =['longitude', 'latitude', 'Datetime_BJT', 'Datetime_UTC', 'Rainfall_Mean_mmh', 'Rainfall_05h_mm']
    df_res = df_res[cols_to_keep].rename(columns={'longitude': 'grid_lon', 'latitude': 'grid_lat'})
    
    # 按照 经度-纬度-时间 排序
    df_res = df_res.sort_values(['grid_lon', 'grid_lat', 'Datetime_BJT']).reset_index(drop=True)
    
    return df_res

# =========================================================
# 6. 循环执行提取
# =========================================================
print("\n" + "="*50)
print("🚀 开始请求 GEE 提取【格网级】高频降雨数据...")
print("="*50)

for idx, row in event_windows.iterrows():
    city = str(row['city_clean'])
    evt_id = str(row['Event_ID'])
    start_utc = row['start_time_utc']
    end_utc = row['end_time_utc']
    
    safe_city = city.replace("/", "_").replace("\\", "_")
    out_csv = OUT_DIR / f"Rainfall_Grid_05h_{safe_city}_Event{evt_id}.csv"
    
    if out_csv.exists():
        print(f"[{idx+1}/{len(event_windows)}] 已存在，跳过: {city}")
        continue

    print(f"[{idx+1}/{len(event_windows)}] 正在提取格网数据: {city} (Event: {evt_id})")
    
    # 匹配城市边界
    city_poly = gpd.GeoDataFrame()
    if has_diji:
        city_poly = china_shp[china_shp['地级'].str.contains(city, na=False)]
    if city_poly.empty and has_shengji:
        city_poly = china_shp[china_shp['省级'].str.contains(city, na=False)]
        
    if city_poly.empty:
        print(f"    ⚠️ 未找到城市: {city}，跳过！")
        continue
        
    city_geom = city_poly.geometry.unary_union
    
    try:
        df_grid = extract_gpm_gridded(start_utc, end_utc, city_geom)
        
        if df_grid.empty:
            print(f"    ⚠️ 云端无数据。")
        else:
            df_grid['city_clean'] = city
            df_grid['Event_ID'] = evt_id
            
            # 列重排
            cols =['city_clean', 'Event_ID', 'grid_lon', 'grid_lat', 'Datetime_BJT', 'Rainfall_05h_mm']
            df_grid = df_grid[cols]
            
            df_grid.to_csv(out_csv, index=False, encoding='utf-8-sig')
            n_pixels = len(df_grid[['grid_lon', 'grid_lat']].drop_duplicates())
            print(f"    ✅ 成功! 提取了该市 {n_pixels} 个 10km 网格的降雨序列 (共 {len(df_grid)} 行)。")
            
    except Exception as e:
        print(f"    ❌ 提取失败: {str(e)}")

print("\n🎉 所有事件的格网级降雨序列提取完成！数据保存在:", OUT_DIR)

GEE 初始化成功！

正在读取事件表并计算时间窗口...
测试模式：只提取 Event 1283，共 0 个事件。
成功构建 0 个事件的时间窗口。
正在加载全国省市县行政区划矢量数据(shp)...

🚀 开始请求 GEE 提取【格网级】高频降雨数据...

🎉 所有事件的格网级降雨序列提取完成！数据保存在: D:\SM\zdxrun\datatest\research\数据集\rainfall_gridded_05h


In [5]:
# -*- coding: utf-8 -*-
"""
基于城市行政边界与时间窗口的 GPM 0.5h 【原生格网级】降雨提取脚本
(取消GEE空间重采样平滑，保留极端雨峰的绝对保真度)
"""

import os
from pathlib import Path  # 已修复：补充 Path 库
import pandas as pd
import geopandas as gpd
import ee
import warnings
warnings.filterwarnings('ignore')

# =========================================================
# 0. 测试模式开关 (非常重要)
# =========================================================
# 如果设为 True，则只提取 Event 1283 进行 A/B 测试。
# 测试确认无误后，将此处改为 False 即可提取全量数据！
TEST_MODE = True  
TEST_EVENT_ID = '1283'

# =========================================================
# 1. 网络代理配置 (根据你的梯子软件修改端口)
# =========================================================
os.environ['HTTP_PROXY'] = 'http://YOUR_PROXY_HOST:PORT'
os.environ['HTTPS_PROXY'] = 'http://YOUR_PROXY_HOST:PORT'

# =========================================================
# 2. GEE 初始化
# =========================================================
try:
    ee.Initialize()
    print("GEE 初始化成功！")
except Exception as e:
    print("正在进行 GEE 认证...")
    ee.Authenticate()
    ee.Initialize()
    print("GEE 授权并初始化成功！")

# =========================================================
# 3. 路径配置
# =========================================================
BASE_DIR = Path(r"D:/SM/zdxrun/datatest/research/数据集")

CSV_PATH = BASE_DIR / "screening_outputs" / "spatial_checks" / "Weibo_Flood_Master_V4_SpatialCleaned.csv"
SHP_PATH = r"D:\数据\2023年中国省市县三级行政区划shp\2023年中国省市县三级行政区划shp\2023年地级\2023年初地级矢量.shp"

OUT_DIR = BASE_DIR / "rainfall_gridded_05h"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# 4. 读取事件窗口
# =========================================================
print("\n正在读取事件表并计算时间窗口...")
df = pd.read_csv(CSV_PATH, low_memory=False)

df['开始时间'] = pd.to_datetime(df['开始时间'], errors='coerce')
df['used_time_str'] = pd.to_datetime(df['used_time_str'], errors='coerce')

event_windows = df.groupby(['city_clean', 'Event_ID']).agg(
    start_time_bjt=('开始时间', 'min'),
    end_time_bjt=('used_time_str', 'max'),
    n_points=('clean_lon', 'count')
).reset_index()

event_windows = event_windows.dropna(subset=['start_time_bjt', 'end_time_bjt'])
event_windows['start_time_bjt'] = event_windows['start_time_bjt'] - pd.Timedelta(hours=48)
event_windows['end_time_bjt'] = event_windows['end_time_bjt'] + pd.Timedelta(hours=12)

event_windows['start_time_utc'] = event_windows['start_time_bjt'] - pd.Timedelta(hours=8)
event_windows['end_time_utc'] = event_windows['end_time_bjt'] - pd.Timedelta(hours=8)

# 强制将 Event_ID 转为字符串以防匹配失败
event_windows['Event_ID'] = event_windows['Event_ID'].astype(str)

# 【应用测试模式过滤】
if TEST_MODE:
    event_windows = event_windows[event_windows['Event_ID'] == TEST_EVENT_ID].copy()
    print(f"\n⚠️ 当前处于测试模式，只提取 Event {TEST_EVENT_ID}，共 {len(event_windows)} 个城市事件。")
else:
    print(f"\n成功构建 {len(event_windows)} 个事件的时间窗口。")

# =========================================================
# 5. 加载行政边界 SHP
# =========================================================
print("正在加载全国行政区划矢量数据(shp)...")
try:
    china_shp = gpd.read_file(SHP_PATH, encoding='utf-8')
except Exception:
    china_shp = gpd.read_file(SHP_PATH, encoding='gbk')

if china_shp.crs.to_string() != 'EPSG:4326':
    china_shp = china_shp.to_crs(epsg=4326)

has_diji = '地级' in china_shp.columns
has_shengji = '省级' in china_shp.columns

# =========================================================
# 6. 定义格网降雨提取核心函数 (原生保真升级版)
# =========================================================
def extract_gpm_gridded(start_utc, end_utc, city_poly):
    simplified_poly = city_poly.simplify(0.01)
    if simplified_poly.geom_type == 'Polygon':
        coords =[list(simplified_poly.exterior.coords)]
        ee_geom = ee.Geometry.Polygon(coords)
    elif simplified_poly.geom_type == 'MultiPolygon':
        coords = [[list(poly.exterior.coords)] for poly in simplified_poly.geoms]
        ee_geom = ee.Geometry.MultiPolygon(coords)
    else:
        raise ValueError("不支持的几何类型")

    dataset = ee.ImageCollection("NASA/GPM_L3/IMERG_V06") \
                .filterBounds(ee_geom) \
                .filterDate(start_utc.strftime('%Y-%m-%dT%H:%M:%S'), 
                            end_utc.strftime('%Y-%m-%dT%H:%M:%S')) \
                .select('precipitationCal')

    # 裁剪到边界
    dataset_clipped = dataset.map(lambda img: img.clip(ee_geom))

    # 【终极保真杀招修复】：直接传入 GPM 卫星的 0.1度 原生标准仿射变换矩阵
    # 彻底拒绝 GEE 的自动重采样平滑，绕过 API 字符串报错
    native_transform =[0.1, 0.0, -180.0, 0.0, -0.1, 90.0]

    region_data = dataset_clipped.getRegion(
        geometry=ee_geom, 
        crs='EPSG:4326', 
        crsTransform=native_transform
    ).getInfo()
    
    
    if not region_data or len(region_data) <= 1:
        return pd.DataFrame()
    
    header = region_data[0]
    data_rows = region_data[1:]
    df_res = pd.DataFrame(data_rows, columns=header)
    
    df_res = df_res.dropna(subset=['precipitationCal']).copy()
    if df_res.empty:
        return df_res
        
    df_res['Datetime_UTC'] = pd.to_datetime(df_res['time'], unit='ms')
    df_res['Datetime_BJT'] = df_res['Datetime_UTC'] + pd.Timedelta(hours=8)
    
    df_res['Rainfall_Mean_mmh'] = df_res['precipitationCal']
    df_res['Rainfall_05h_mm'] = df_res['Rainfall_Mean_mmh'] / 2.0
    
    cols_to_keep =['longitude', 'latitude', 'Datetime_BJT', 'Datetime_UTC', 'Rainfall_Mean_mmh', 'Rainfall_05h_mm']
    df_res = df_res[cols_to_keep].rename(columns={'longitude': 'grid_lon', 'latitude': 'grid_lat'})
    df_res = df_res.sort_values(['grid_lon', 'grid_lat', 'Datetime_BJT']).reset_index(drop=True)
    
    return df_res

# =========================================================
# 7. 循环执行提取
# =========================================================
print("\n" + "="*50)
print("🚀 开始请求 GEE 提取【原生保真格网】降雨数据...")
print("="*50)

for idx, row in event_windows.iterrows():
    city = str(row['city_clean'])
    evt_id = str(row['Event_ID'])
    start_utc = row['start_time_utc']
    end_utc = row['end_time_utc']
    
    safe_city = city.replace("/", "_").replace("\\", "_")
    out_csv = OUT_DIR / f"Rainfall_Grid_05h_{safe_city}_Event{evt_id}.csv"
    
    # 强制覆盖旧文件，以确保测试结果是全新的
    if out_csv.exists() and TEST_MODE:
        out_csv.unlink()
        print(f"[{idx+1}/{len(event_windows)}] 已删除旧文件，准备重新提取: {city} (Event: {evt_id})")

    elif out_csv.exists() and not TEST_MODE:
        print(f"[{idx+1}/{len(event_windows)}] 已存在，跳过: {city} (Event: {evt_id})")
        continue

    print(f"[{idx+1}/{len(event_windows)}] 正在提取格网数据: {city} (Event: {evt_id})")
    
    city_poly = gpd.GeoDataFrame()
    if has_diji:
        city_poly = china_shp[china_shp['地级'].str.contains(city, na=False)]
    if city_poly.empty and has_shengji:
        city_poly = china_shp[china_shp['省级'].str.contains(city, na=False)]
        
    if city_poly.empty:
        print(f"    ⚠️ 未找到城市边界: {city}，跳过！")
        continue
        
    city_geom = city_poly.geometry.unary_union
    
    try:
        df_grid = extract_gpm_gridded(start_utc, end_utc, city_geom)
        
        if df_grid.empty:
            print(f"    ⚠️ 云端无数据。")
        else:
            df_grid['city_clean'] = city
            df_grid['Event_ID'] = evt_id
            
            cols =['city_clean', 'Event_ID', 'grid_lon', 'grid_lat', 'Datetime_BJT', 'Rainfall_05h_mm']
            df_grid = df_grid[cols]
            
            df_grid.to_csv(out_csv, index=False, encoding='utf-8-sig')
            n_pixels = len(df_grid[['grid_lon', 'grid_lat']].drop_duplicates())
            print(f"    ✅ 成功! 提取了该市 {n_pixels} 个原生网格的降雨序列 (共 {len(df_grid)} 行)。")
            
    except Exception as e:
        print(f"    ❌ 提取失败: {str(e)}")

print("\n🎉 提取完成！数据保存在:", OUT_DIR)

GEE 初始化成功！

正在读取事件表并计算时间窗口...

⚠️ 当前处于测试模式，只提取 Event 1283，共 1 个城市事件。
正在加载全国行政区划矢量数据(shp)...

🚀 开始请求 GEE 提取【原生保真格网】降雨数据...
[5/1] 正在提取格网数据: 三亚市 (Event: 1283)
    ✅ 成功! 提取了该市 16 个原生网格的降雨序列 (共 3152 行)。

🎉 提取完成！数据保存在: D:\SM\zdxrun\datatest\research\数据集\rainfall_gridded_05h


In [7]:
# -*- coding: utf-8 -*-
"""
新旧降雨数据源差异比对：点位聚合 (旧) vs 行政区格网 (新)
"""

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# =========================================================
# 1. 路径配置
# =========================================================
BASE_DIR = Path(r"D:/SM/zdxrun/datatest/research/数据集")

# 旧数据：基于内涝点提取的时间序列
OLD_RAIN_FILE = BASE_DIR / "All_Events_Rainfall_TimeSeries.csv"

# 新数据：GEE提取的城市格网时间序列文件夹
NEW_RAIN_DIR = BASE_DIR / "rainfall_gridded_05h"

OUT_DIR = BASE_DIR / "screening_outputs" / "rainfall_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# 2. 读取旧数据
# =========================================================
print("正在读取旧版降水数据...")
df_old = pd.read_csv(OLD_RAIN_FILE, low_memory=False)
df_old['Datetime_UTC'] = pd.to_datetime(df_old['Datetime_UTC'], errors='coerce')
df_old = df_old.dropna(subset=['Datetime_UTC', 'Event_ID'])
df_old['Event_ID'] = df_old['Event_ID'].astype(str)

# 确保时间转换为北京时间 (BJT = UTC + 8) 以便直观比对
df_old['Datetime_BJT'] = df_old['Datetime_UTC'] + pd.Timedelta(hours=8)

# =========================================================
# 3. 扫描新数据并挑选对比事件
# =========================================================
print("正在扫描新版格网降水数据...")
new_files = list(NEW_RAIN_DIR.glob("Rainfall_Grid_05h_*.csv"))

if not new_files:
    raise FileNotFoundError("未在 rainfall_gridded_05h 目录中找到新版降水数据！")

# 挑选前 2 个文件进行可视化对比
sample_files = new_files[:2]

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

for file_path in sample_files:
    # 从文件名解析城市和事件ID (Rainfall_Grid_05h_城市_EventXXX.csv)
    filename = file_path.stem
    parts = filename.split("_Event")
    city_name = parts[0].replace("Rainfall_Grid_05h_", "")
    event_id = parts[1]
    
    print(f"\n正在比对事件: {city_name} (Event: {event_id})")
    
    # 提取该事件的新旧数据
    old_sub = df_old[df_old['Event_ID'] == event_id].copy()
    if old_sub.empty:
        print(f"  ⚠️ 旧数据中未找到 Event_ID {event_id}，跳过该事件。")
        continue
        
    old_sub = old_sub.sort_values('Datetime_BJT')
    
    new_sub = pd.read_csv(file_path)
    new_sub['Datetime_BJT'] = pd.to_datetime(new_sub['Datetime_BJT'])
    
    # 【核心转换】：由于新数据存的是 Rainfall_05h_mm (半小时真实雨量)，
    # 为了和旧数据里的 Rainfall_Max_mmh (雨强, 毫米/小时) 对齐，我们需要乘以 2
    new_sub['Rainfall_Intensity_mmh'] = new_sub['Rainfall_05h_mm'] * 2.0
    
    # 对新格网数据进行空间聚合统计 (同一时刻，全城所有格网的最大值、均值、最小值)
    new_agg = new_sub.groupby('Datetime_BJT').agg(
        Grid_Max_mmh=('Rainfall_Intensity_mmh', 'max'),
        Grid_Mean_mmh=('Rainfall_Intensity_mmh', 'mean'),
        Grid_Min_mmh=('Rainfall_Intensity_mmh', 'min')
    ).reset_index()
    
    # 合并新旧数据以对齐时间轴
    merged = pd.merge(new_agg, old_sub[['Datetime_BJT', 'Rainfall_Max_mmh', 'Rainfall_Mean_mmh']], 
                      on='Datetime_BJT', how='inner')
    
    if merged.empty:
        print(f"  ⚠️ 新旧数据在时间轴上没有交集，跳过。")
        continue

    # =========================================================
    # 4. 绘图：新旧差异可视化
    # =========================================================
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # 画新数据：全城格网的空间异质性区间 (Shaded area representing spatial variance)
    ax.fill_between(merged['Datetime_BJT'], merged['Grid_Min_mmh'], merged['Grid_Max_mmh'], 
                    color='lightblue', alpha=0.4, label='New Gridded: Spatial Min-Max Range')
    
    # 画新数据：全城格网最大值
    ax.plot(merged['Datetime_BJT'], merged['Grid_Max_mmh'], 
            color='blue', linestyle='-', linewidth=2, label='New Gridded: City-wide Max')
    
    # 画旧数据：基于内涝点提取的最大值
    if 'Rainfall_Max_mmh' in merged.columns:
        ax.plot(merged['Datetime_BJT'], merged['Rainfall_Max_mmh'], 
                color='red', linestyle='--', linewidth=2, label='Old Data: Flood-point Max')
                
    # 画旧数据：基于内涝点提取的均值
    if 'Rainfall_Mean_mmh' in merged.columns:
        ax.plot(merged['Datetime_BJT'], merged['Rainfall_Mean_mmh'], 
                color='orange', linestyle='-.', linewidth=2, label='Old Data: Flood-point Mean')

    ax.set_title(f"Rainfall Data Comparison: {city_name} (Event: {event_id})", fontsize=15)
    ax.set_xlabel("Time (Beijing Time)")
    ax.set_ylabel("Rainfall Intensity (mm/h)")
    ax.legend(loc="upper right", frameon=True)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    out_file = OUT_DIR / f"Comparison_{city_name}_Event{event_id}.png"
    plt.savefig(out_file, dpi=300)
    plt.close()
    
    print(f"  ✅ 对比图已生成: {out_file.name}")
    
    # 打印简要统计
    corr_max = merged['Grid_Max_mmh'].corr(merged['Rainfall_Max_mmh'])
    print(f"  -> 新旧最大雨强相关系数 (Pearson's r): {corr_max:.3f}")

print("\n🎉 对比完成！请前往 screening_outputs/rainfall_comparison 查看图片。")

正在读取旧版降水数据...
正在扫描新版格网降水数据...

正在比对事件: 三亚市 (Event: 1096)
  ✅ 对比图已生成: Comparison_三亚市_Event1096.png
  -> 新旧最大雨强相关系数 (Pearson's r): 0.849

正在比对事件: 三亚市 (Event: 1283)
  ✅ 对比图已生成: Comparison_三亚市_Event1283.png
  -> 新旧最大雨强相关系数 (Pearson's r): 0.762

🎉 对比完成！请前往 screening_outputs/rainfall_comparison 查看图片。


In [9]:
# -*- coding: utf-8 -*-
"""
更新事件级降水时间序列：将 GPM 格网数据聚合为全城宏观时间序列
(已修复列名缺失问题，自动反推 UTC 时间与降雨强度)
"""

from pathlib import Path
import pandas as pd

# =========================================================
# 1. 路径配置
# =========================================================
BASE_DIR = Path(r"D:/SM/zdxrun/datatest/research/数据集")

# 输入：刚才提取的所有 0.5h 格网降水 CSV 所在的文件夹
GRIDDED_DIR = BASE_DIR / "rainfall_gridded_05h"

# 输出：更新后的全新事件级时间序列文件
OUT_FILE = BASE_DIR / "All_Events_Rainfall_TimeSeries_Gridded.csv"

# =========================================================
# 2. 遍历并聚合数据
# =========================================================
file_list = list(GRIDDED_DIR.glob("Rainfall_Grid_05h_*.csv"))
print(f"🔍 找到 {len(file_list)} 个格网降雨文件，开始聚合为事件级时间序列...\n")

all_events_data =[]
processed_count = 0

for file_path in file_list:
    df = pd.read_csv(file_path)
    if df.empty:
        continue
        
    # 1. 时间格式转换
    df['Datetime_BJT'] = pd.to_datetime(df['Datetime_BJT'])
    
    # 2. 找回缺失的列：反推 UTC 时间 和 小时降雨强度 (mm/h)
    df['Datetime_UTC'] = df['Datetime_BJT'] - pd.Timedelta(hours=8)
    # Rainfall_05h_mm 是半小时下到地面的真实雨量，乘以 2 就是降雨强度(mm/h)
    df['Rainfall_Intensity_mmh'] = df['Rainfall_05h_mm'] * 2.0 
        
    # 3. 按事件和时间分组，计算全城网格的统计量
    agg_df = df.groupby(['Event_ID', 'city_clean', 'Datetime_UTC', 'Datetime_BJT']).agg(
        Rainfall_Max_mmh=('Rainfall_Intensity_mmh', 'max'),
        Rainfall_Mean_mmh=('Rainfall_Intensity_mmh', 'mean'),
        Rainfall_Min_mmh=('Rainfall_Intensity_mmh', 'min')
    ).reset_index()
    
    all_events_data.append(agg_df)
    processed_count += 1
    
    if processed_count % 100 == 0:
        print(f"  ...已处理 {processed_count} 个事件...")

# =========================================================
# 3. 合并与保存
# =========================================================
if not all_events_data:
    raise ValueError("没有找到有效的降水数据，请检查输入目录！")

print("\n🚀 正在合并所有数据...")
final_df = pd.concat(all_events_data, ignore_index=True)

# 确保 Event_ID 为字符串格式，并按事件和时间排序
final_df['Event_ID'] = final_df['Event_ID'].astype(str)
final_df = final_df.sort_values(['Event_ID', 'Datetime_UTC']).reset_index(drop=True)

# 保存文件
final_df.to_csv(OUT_FILE, index=False, encoding='utf-8-sig')

print("="*50)
print(f"🎉 更新大功告成！")
print(f"共聚合了 {processed_count} 个事件。")
print(f"生成的全新时间序列文件已保存至：\n{OUT_FILE}")
print("="*50)

🔍 找到 2336 个格网降雨文件，开始聚合为事件级时间序列...

  ...已处理 100 个事件...
  ...已处理 200 个事件...
  ...已处理 300 个事件...
  ...已处理 400 个事件...
  ...已处理 500 个事件...
  ...已处理 600 个事件...
  ...已处理 700 个事件...
  ...已处理 800 个事件...
  ...已处理 900 个事件...
  ...已处理 1000 个事件...
  ...已处理 1100 个事件...
  ...已处理 1200 个事件...
  ...已处理 1300 个事件...
  ...已处理 1400 个事件...
  ...已处理 1500 个事件...
  ...已处理 1600 个事件...
  ...已处理 1700 个事件...
  ...已处理 1800 个事件...
  ...已处理 1900 个事件...
  ...已处理 2000 个事件...
  ...已处理 2100 个事件...
  ...已处理 2200 个事件...
  ...已处理 2300 个事件...

🚀 正在合并所有数据...
🎉 更新大功告成！
共聚合了 2336 个事件。
生成的全新时间序列文件已保存至：
D:\SM\zdxrun\datatest\research\数据集\All_Events_Rainfall_TimeSeries_Gridded.csv
